In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from pykalman import KalmanFilter
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Conv1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ================================================
# ROBUST INFLATION FORECASTING WITH KALMAN FILTERING
# Scientific approach using leading macroeconomic indicators
# ================================================

print("🚀 ROBUST INFLATION FORECASTING WITH KALMAN FILTERING")
print("="*70)
print("✅ NO inflation as predictor (avoiding data leakage)")
print("✅ Using Kalman filter for optimal state estimation")  
print("✅ Multiple evaluation metrics (MSE, MAE, R², MAPE)")
print("✅ Scientific methodology for policy applications")
print("✅ Leading macroeconomic indicators from Fed data")
print("="*70)

# ================================================
# 1. SCIENTIFIC SETUP - LEADING MACROECONOMIC INDICATORS
# ================================================

def load_and_document_data(file_path):
    """
    Load inflation forecasting data with leading macroeconomic indicators
    
    Leading Indicators for Kalman Filter Analysis:
    - USM2: Money Supply M2 (monetary policy stance)
    - USPPIYY: Producer Price Index YoY (cost-push inflation signal)  
    - INDPRO: Industrial Production Index (economic activity indicator)
    - UNRATE: Unemployment Rate (labor market conditions)
    - USOIL: Oil Prices (commodity/supply shock indicator)
    
    Target:
    - Annual Inflation Rate (what central banks need to forecast)
    
    Note: Kalman filter will estimate optimal hidden states from these observables
    """
    print("\n📊 LOADING MACROECONOMIC INDICATORS FOR KALMAN FILTERING")
    print("-" * 50)
    
    try:
        data = pd.read_csv(file_path)
        print(f"✅ Data loaded: {data.shape[0]} observations, {data.shape[1]} features")
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        print("📁 Please check the file path and ensure the CSV file exists")
        return None
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        return None
    
    # Handle missing values
    missing_before = data.isnull().sum().sum()
    print(f"📋 Missing values before: {missing_before}")
    
    if missing_before > 0:
        data = data.interpolate(method='linear')
        data = data.fillna(method='ffill').fillna(method='bfill')
        
    missing_after = data.isnull().sum().sum()
    print(f"✅ Missing values after interpolation: {missing_after}")
    
    return data

# ================================================
# 2. SCIENTIFIC VARIABLE DEFINITION - SAME AS HP FILTER CODE
# ================================================

# LEADING MACROECONOMIC INDICATORS (no inflation as predictor)
LEADING_INDICATORS = [
    'USM2',      # Money Supply M2 - Monetary policy stance
    'USPPIYY',   # Producer Price Index YoY - Cost pressures  
    'INDPRO',    # Industrial Production - Economic activity
    'UNRATE',    # Unemployment Rate - Labor market conditions
    'USOIL'      # Oil Prices - Supply shocks and commodity inflation
]

TARGET_VARIABLE = 'Annual Inflation Rate'

print(f"\n🎯 KALMAN FILTER RESEARCH DESIGN:")
print(f"Leading Indicators (Observable States): {LEADING_INDICATORS}")
print(f"Target Variable (Hidden State to Forecast): {TARGET_VARIABLE}")
print(f"\n💡 Research Question:")
print(f"Can Kalman filtering enhance ML/DL inflation forecasting")
print(f"by optimally estimating economic states from Fed indicators?")

# ================================================
# 3. MODEL CONFIGURATIONS (MAINTAINED FROM ORIGINAL)
# ================================================

MODEL_CONFIGS = {
    'LSTM': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: LSTM(50) → LSTM(50) → Dense(1)',
        'optimal_for': 'Long-term temporal dependencies in economic data'
    },
    'GRU': {
        'units': 50,
        'activation': 'tanh', 
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: GRU(50) → GRU(50) → Dense(1)',
        'optimal_for': 'Computational efficiency with temporal memory'
    },
    'CNN': {
        'filters': 64,
        'kernel_size': 2,
        'activation': 'relu',
        'dense_units': 50,
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'input_type': '1D convolution',
        'architecture': 'Conv1D(64) → Flatten → Dense(50) → Dense(1)',
        'optimal_for': 'Local pattern recognition in economic time series'
    },
    'RNN': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential: RNN(50) → RNN(50) → Dense(1)',
        'optimal_for': 'Basic sequential pattern recognition'
    },
    'ANN': {
        'units': 50,
        'activation': 'relu',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Dense(50) → Dense(50) → Dense(1)',
        'optimal_for': 'Non-linear relationships in flattened sequences'
    }
}

print(f"\n🤖 MODEL ARCHITECTURES FOR KALMAN-FILTERED FED DATA:")
for model_name, config in MODEL_CONFIGS.items():
    print(f"   {model_name}: {config['architecture']}")
    print(f"     └─ Optimal for: {config['optimal_for']}")

# ================================================
# 4. ENHANCED MULTIDIMENSIONAL KALMAN FILTER IMPLEMENTATION
# ================================================

def apply_multidimensional_kalman_filter(data, n_iter_em=10, burn_in=10):
    """
    Apply Multidimensional Kalman Filter for optimal state estimation of economic indicators
    
    This implementation uses a multivariate approach where:
    - All indicators are filtered simultaneously to capture cross-correlations
    - EM algorithm estimates optimal covariance matrices automatically
    - Smoothing is used instead of filtering for more stable state estimates
    - Identity matrices are used for transition and observation models
    
    Parameters:
    -----------
    data : DataFrame
        Observable economic time series (all indicators)
    n_iter_em : int
        Number of iterations for EM algorithm
    burn_in : int
        Number of initial observations to discard
            
    Returns:
    --------
    np.array : Kalman-smoothed optimal economic state estimates
    """
    try:
        n_dim = data.shape[1]
        obs = data.values  # Convert to numpy for Kalman filter
        
        print(f"      🔧 Applying Multidimensional Kalman filter (n_dim={n_dim}, EM iterations={n_iter_em})")
        
        # Initialize Kalman Filter with multivariate configuration
        kf = KalmanFilter(
            n_dim_obs=n_dim,          # Multivariate observations
            n_dim_state=n_dim,        # Multivariate state
            transition_matrices=np.eye(n_dim),      # Identity for random walk model
            observation_matrices=np.eye(n_dim),     # Direct observation model
            initial_state_mean=np.zeros(n_dim),     # Start with zeros
            initial_state_covariance=np.eye(n_dim) * 1e3  # High initial uncertainty
        )
        
        # Use EM to estimate optimal parameters from data
        print(f"      📊 Estimating optimal covariances via EM algorithm...")
        kf = kf.em(obs, n_iter=n_iter_em)
        
        # Apply smoothing (instead of filtering) for optimal retrospective estimates
        print(f"      🔄 Applying Kalman smoothing for optimal retrospective estimates...")
        state_means, state_covs = kf.smooth(obs)
        
        # Discard burn-in period
        if burn_in > 0 and burn_in < state_means.shape[0]:
            print(f"      🔥 Discarding {burn_in} burn-in observations...")
            state_means = state_means[burn_in:, :]
            
        return state_means
            
    except Exception as e:
        print(f"      ⚠️ Multidimensional Kalman Filter error: {e}")
        print(f"      ↳ Returning original data (fallback)")
        return data.values[burn_in:] if burn_in > 0 else data.values

# ================================================
# 5. PREPROCESSING FUNCTIONS (UPDATED SEQUENCE)
# ================================================

def create_sequences(data, seq_length):
    """
    Create sequences for time series prediction from Kalman-filtered Fed data
    
    Input structure after Kalman filtering:
    - Each sequence contains seq_length time steps of Kalman-filtered Fed indicators
    - Features: Kalman-filtered leading indicators (USM2, USPPIYY, INDPRO, UNRATE, USOIL)
    - Target: Annual Inflation Rate at time t+1
    
    This maintains temporal structure while using optimal state estimates
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length, :-1])  # Kalman-filtered Fed indicators
        y.append(data[i + seq_length, -1])     # Target: inflation at t+1
    return np.array(X), np.array(y)

# ================================================
# 6. MODEL BUILDERS (MAINTAINED WITH ENHANCEMENTS)
# ================================================

def build_lstm_model(input_shape):
    """Build LSTM model optimized for Kalman-filtered Fed economic indicators"""
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))  # Regularization for economic volatility
    model.add(LSTM(50, activation='tanh'))
    model.add(Dropout(0.2))  # Prevent overfitting on economic patterns
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_gru_model(input_shape):
    """Build GRU model optimized for Kalman-filtered economic time series"""
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(GRU(50, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_cnn_model(input_shape):
    """Build CNN model for 1D convolution on Kalman-filtered Fed sequences"""
    model = Sequential()
    model.add(Conv1D(64, kernel_size=2, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_rnn_model(input_shape):
    """Build RNN model for Kalman-filtered economic time series"""
    model = Sequential()
    model.add(SimpleRNN(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(SimpleRNN(50, activation='tanh'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_ann_model(input_shape):
    """Build ANN model for flattened Kalman-filtered Fed sequences"""
    model = Sequential()
    model.add(Dense(50, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# ================================================
# 7. COMPREHENSIVE METRICS
# ================================================

def calculate_comprehensive_metrics(y_true, y_pred):
    """Calculate comprehensive evaluation metrics for Kalman-enhanced inflation forecasting"""
    return {
        'mse': mean_squared_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

# ================================================
# 8. ML BASELINE MODELS
# ================================================

def train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test):
    """Train ML baseline models on Kalman-filtered Fed indicators"""
    print("   🌳 Training ML baselines on Kalman-filtered Fed data...")
    ml_results = {}
    
    try:
        # Random Forest
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_flat, y_train)
        rf_pred = rf_model.predict(X_test_flat)
        ml_results['random_forest'] = calculate_comprehensive_metrics(y_test, rf_pred)
        
        # XGBoost
        xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        xgb_model.fit(X_train_flat, y_train)
        xgb_pred = xgb_model.predict(X_test_flat)
        ml_results['xgboost'] = calculate_comprehensive_metrics(y_test, xgb_pred)
        
        print(f"      ✅ Random Forest: MSE={ml_results['random_forest']['mse']:.6f}")
        print(f"      ✅ XGBoost: MSE={ml_results['xgboost']['mse']:.6f}")
        
    except Exception as e:
        print(f"      ⚠️ ML baseline error: {e}")
        
    return ml_results

# ================================================
# 9. SEQUENCE LENGTH SENSITIVITY ANALYSIS
# ================================================

def sequence_length_sensitivity_analysis(data, indicators, target, burn_in=10, n_iter_em=10):
    """
    Perform sensitivity analysis for optimal sequence length selection
    
    This function tests different sequence lengths to determine which 
    one provides optimal performance for Kalman-filtered time series.
    """
    print("\n📏 SEQUENCE LENGTH SENSITIVITY ANALYSIS")
    print("-" * 50)
    
    sequence_lengths = [30, 50, 70]
    results = []
    
    try:
        # Prepare dataset with selected indicators + target
        selected_features = indicators + [target]
        selected_data = data[selected_features].copy()
        
        # Apply multivariate Kalman filter
        print(f"   🔧 Applying multivariate Kalman filter to {len(indicators)} indicators...")
        filtered_data = apply_multidimensional_kalman_filter(
            selected_data, 
            n_iter_em=n_iter_em, 
            burn_in=burn_in
        )
        
        # Normalize filtered data
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(filtered_data)
        
        # Test each sequence length
        for seq_length in sequence_lengths:
            print(f"\n   📏 Testing sequence length: {seq_length}")
            
            # Create sequences
            X, y = create_sequences(scaled_data, seq_length)
            
            # Split data
            split_idx = int(len(X) * 0.8)
            X_train, X_test = X[:split_idx], X[split_idx:]
            y_train, y_test = y[:split_idx], y[split_idx:]
            
            # Prepare LSTM (as representative model)
            input_shape = (seq_length, X_train.shape[2])
            model = build_lstm_model(input_shape)
            
            # Train model
            early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            history = model.fit(
                X_train, y_train,
                epochs=100,
                batch_size=32,
                validation_split=0.2,
                verbose=0,
                callbacks=[early_stopping]
            )
            
            # Evaluate
            y_pred = model.predict(X_test, verbose=0).flatten()
            metrics = calculate_comprehensive_metrics(y_test, y_pred)
            
            print(f"      ✅ Sequence length {seq_length}: MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")
            
            # Store results
            results.append({
                'seq_length': seq_length,
                **metrics
            })
        
        # Find optimal sequence length
        optimal_seq_length = min(results, key=lambda x: x['mse'])['seq_length']
        print(f"\n   🏆 Optimal sequence length: {optimal_seq_length}")
        print(f"      Best MSE: {min(result['mse'] for result in results):.6f}")
        print(f"      Best R²: {max(result['r2'] for result in results):.4f}")
        
        return optimal_seq_length, results
        
    except Exception as e:
        print(f"   ❌ Sensitivity analysis failed: {e}")
        return 50, []  # Default to 50 if analysis fails

# ================================================
# 10. WALK-FORWARD VALIDATION
# ================================================

def perform_walk_forward_validation(data, indicators, target, seq_length=50, folds=5, burn_in=10, n_iter_em=10):
    """
    Perform walk-forward validation for temporal stability assessment
    
    This function implements a time series cross-validation approach where
    multiple consecutive train-test splits are evaluated to verify model
    robustness across different time periods.
    """
    print("\n🚶 WALK-FORWARD VALIDATION FOR KALMAN FILTER")
    print("-" * 50)
    
    try:
        # Prepare dataset with selected indicators + target
        selected_features = indicators + [target]
        selected_data = data[selected_features].copy()
        
        # Apply multivariate Kalman filter
        print(f"   🔧 Applying multivariate Kalman filter for validation...")
        filtered_data = apply_multidimensional_kalman_filter(
            selected_data, 
            n_iter_em=n_iter_em, 
            burn_in=burn_in
        )
        
        # Normalize filtered data
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(filtered_data)
        
        # Create sequences
        X, y = create_sequences(scaled_data, seq_length)
        
        # Prepare folds (time-based splits)
        fold_size = len(X) // folds
        results = []
        
        # For each fold
        for fold in range(folds):
            print(f"\n   📂 Fold {fold+1}/{folds}")
            
            # Calculate indices for this fold
            test_start = fold * fold_size
            test_end = (fold + 1) * fold_size if fold < folds - 1 else len(X)
            
            # Split data
            X_train = np.concatenate([X[:test_start], X[test_end:]])
            y_train = np.concatenate([y[:test_start], y[test_end:]])
            X_test = X[test_start:test_end]
            y_test = y[test_start:test_end]
            
            # Prepare model (Random Forest as robust baseline)
            X_train_flat = X_train.reshape(X_train.shape[0], -1)
            X_test_flat = X_test.reshape(X_test.shape[0], -1)
            
            rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
            rf_model.fit(X_train_flat, y_train)
            
            # Evaluate
            y_pred = rf_model.predict(X_test_flat)
            metrics = calculate_comprehensive_metrics(y_test, y_pred)
            
            print(f"      ✅ MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")
            
            # Store results
            results.append({
                'fold': fold + 1,
                **metrics
            })
        
        # Calculate average and stddev across folds
        avg_mse = np.mean([r['mse'] for r in results])
        std_mse = np.std([r['mse'] for r in results])
        avg_r2 = np.mean([r['r2'] for r in results])
        std_r2 = np.std([r['r2'] for r in results])
        
        print(f"\n   📊 Validation Results:")
        print(f"      Average MSE: {avg_mse:.6f} ± {std_mse:.6f}")
        print(f"      Average R²: {avg_r2:.4f} ± {std_r2:.4f}")
        print(f"      Consistency: {'High' if std_mse < 0.01 and std_r2 < 0.1 else 'Medium' if std_mse < 0.05 and std_r2 < 0.2 else 'Low'}")
        
        return results
        
    except Exception as e:
        print(f"   ❌ Walk-forward validation failed: {e}")
        return []

# ================================================
# 11. EM PARAMETER SENSITIVITY ANALYSIS
# ================================================

def em_parameter_sensitivity_analysis(data, indicators, target, seq_length=50, burn_in=10):
    """
    Analyze sensitivity to EM algorithm iterations and burn-in period
    
    This tests different configurations of EM iterations and burn-in 
    periods to find optimal parameters for the multivariate Kalman filter.
    """
    print("\n🔍 EM PARAMETER SENSITIVITY ANALYSIS")
    print("-" * 50)
    
    # Parameter grids
    em_iters = [5, 10, 15]
    burn_in_values = [5, 10, 15]
    
    results = []
    
    try:
        # Prepare dataset with selected indicators + target
        selected_features = indicators + [target]
        selected_data = data[selected_features].copy()
        
        for n_iter_em in em_iters:
            for burn_in_val in burn_in_values:
                print(f"\n   📊 Testing EM iterations={n_iter_em}, burn-in={burn_in_val}")
                
                # Apply multivariate Kalman filter with these parameters
                filtered_data = apply_multidimensional_kalman_filter(
                    selected_data, 
                    n_iter_em=n_iter_em, 
                    burn_in=burn_in_val
                )
                
                # Normalize filtered data
                scaler = MinMaxScaler()
                scaled_data = scaler.fit_transform(filtered_data)
                
                # Create sequences
                X, y = create_sequences(scaled_data, seq_length)
                
                # Split data
                split_idx = int(len(X) * 0.8)
                X_train, X_test = X[:split_idx], X[split_idx:]
                y_train, y_test = y[:split_idx], y[split_idx:]
                
                # Prepare model (Random Forest as stable baseline)
                X_train_flat = X_train.reshape(X_train.shape[0], -1)
                X_test_flat = X_test.reshape(X_test.shape[0], -1)
                
                rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
                rf_model.fit(X_train_flat, y_train)
                
                # Evaluate
                y_pred = rf_model.predict(X_test_flat)
                metrics = calculate_comprehensive_metrics(y_test, y_pred)
                
                print(f"      ✅ MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")
                
                # Store results
                results.append({
                    'n_iter_em': n_iter_em,
                    'burn_in': burn_in_val,
                    **metrics
                })
        
        # Find optimal parameters
        optimal_params = min(results, key=lambda x: x['mse'])
        print(f"\n   🏆 Optimal parameters:")
        print(f"      EM iterations: {optimal_params['n_iter_em']}")
        print(f"      Burn-in: {optimal_params['burn_in']}")
        print(f"      Best MSE: {optimal_params['mse']:.6f}")
        print(f"      Best R²: {optimal_params['r2']:.4f}")
        
        return optimal_params
        
    except Exception as e:
        print(f"   ❌ Parameter sensitivity analysis failed: {e}")
        return {'n_iter_em': 10, 'burn_in': 10}  # Default if analysis fails

# ================================================
# 12. MAIN ANALYSIS - MULTIDIMENSIONAL KALMAN FILTERING + FED INDICATORS
# ================================================

def run_kalman_fed_inflation_forecasting():
    """
    Run comprehensive inflation forecasting with multidimensional Kalman filtering on Fed indicators
    """
    
    # Load data
    file_path = r"C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\Data\Inflation_Parameters.csv"
    print(f"📁 Loading Fed economic indicators from: {file_path}")
    
    merged_data = load_and_document_data(file_path)
    
    if merged_data is None:
        print("❌ Cannot proceed without valid Fed data")
        return []
    
    # Verify required columns exist
    required_columns = LEADING_INDICATORS + [TARGET_VARIABLE]
    missing_columns = [col for col in required_columns if col not in merged_data.columns]
    
    if missing_columns:
        print(f"❌ Missing required Fed indicators: {missing_columns}")
        print(f"📋 Available columns: {list(merged_data.columns)}")
        return []
    else:
        print(f"✅ All required Fed indicators found in dataset")
    
    # First, run sequence length sensitivity analysis
    optimal_seq_length, length_results = sequence_length_sensitivity_analysis(
        merged_data, 
        LEADING_INDICATORS[:2],  # Use a subset for efficiency 
        TARGET_VARIABLE
    )
    
    # Then, run parameter sensitivity analysis
    optimal_params = em_parameter_sensitivity_analysis(
        merged_data,
        LEADING_INDICATORS[:2],  # Use a subset for efficiency
        TARGET_VARIABLE,
        seq_length=optimal_seq_length
    )
    
    # Finally, perform walk-forward validation with optimal parameters
    validation_results = perform_walk_forward_validation(
        merged_data,
        LEADING_INDICATORS[:2],  # Use a subset for efficiency
        TARGET_VARIABLE,
        seq_length=optimal_seq_length,
        folds=5,
        burn_in=optimal_params['burn_in'],
        n_iter_em=optimal_params['n_iter_em']
    )
    
    # Now proceed with main analysis using optimal parameters
    results = []
    seq_length = optimal_seq_length
    n_iter_em = optimal_params['n_iter_em']
    burn_in = optimal_params['burn_in']
    
    # Test both normalization methods
    scalers = {
        'MinMax': MinMaxScaler(),
        'Standard': StandardScaler()
    }
    
    print(f"\n🔬 MULTIDIMENSIONAL KALMAN FILTER FED INFLATION FORECASTING EXPERIMENT")
    print(f"Fed Leading indicators: {LEADING_INDICATORS}")
    print(f"Target: {TARGET_VARIABLE}")
    print(f"Optimal sequence length: {seq_length}")
    print(f"Optimal EM iterations: {n_iter_em}")
    print(f"Optimal burn-in: {burn_in}")
    print(f"Normalization methods: {list(scalers.keys())}")
    print("-" * 70)
    
    # Calculate total experiments
    total_combinations = sum(1 for r in range(1, len(LEADING_INDICATORS) + 1) 
                           for _ in combinations(LEADING_INDICATORS, r))
    total_experiments = total_combinations * len(scalers)
    current_experiment = 0
    
    # Iterate through all combinations of Fed leading indicators
    for r in range(1, len(LEADING_INDICATORS) + 1):
        for combo in combinations(LEADING_INDICATORS, r):
            for scaler_name, scaler in scalers.items():
                current_experiment += 1
                
                print(f"\n📊 Experiment {current_experiment}/{total_experiments}")
                print(f"    Fed Indicators: {combo}")
                print(f"    Normalization: {scaler_name}")
                
                try:
                    # Prepare dataset with ONLY Fed leading indicators + target
                    selected_features = list(combo) + [TARGET_VARIABLE]
                    selected_data = merged_data[selected_features].copy()
                    
                    print(f"   📊 Selected Fed data shape: {selected_data.shape}")
                    
                    # Apply Multidimensional Kalman Filter for optimal state estimation
                    print("   🔧 Applying Multidimensional Kalman Filter to Fed indicators...")
                    filtered_data = apply_multidimensional_kalman_filter(
                        selected_data, 
                        n_iter_em=n_iter_em, 
                        burn_in=burn_in
                    )
                    
                    # Normalize Kalman-filtered data
                    scaled_data = scaler.fit_transform(filtered_data)
                    
                    # Create sequences for time series modeling
                    X, y = create_sequences(scaled_data, seq_length)
                    
                    if len(X) < 20:  # Minimum samples for reliable training
                        print("   ⚠️ Insufficient data for reliable training")
                        continue
                    
                    # Temporal train-test split (crucial for time series)
                    split_idx = int(len(X) * 0.8)
                    X_train, X_test = X[:split_idx], X[split_idx:]
                    y_train, y_test = y[:split_idx], y[split_idx:]
                    
                    print(f"   📊 Data split: Train={len(X_train)}, Test={len(X_test)}")
                    
                    # Prepare input shapes
                    input_shape_seq = (seq_length, X_train.shape[2])
                    X_train_flat = X_train.reshape(X_train.shape[0], -1)
                    X_test_flat = X_test.reshape(X_test.shape[0], -1)
                    input_shape_ann = (X_train_flat.shape[1],)
                    
                    # Early stopping callback
                    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
                    
                    # Train all models
                    models_results = []
                    
                    # Neural Network Models
                    neural_models = [
                        ('LSTM', build_lstm_model, X_train, X_test, input_shape_seq),
                        ('GRU', build_gru_model, X_train, X_test, input_shape_seq),
                        ('CNN', build_cnn_model, X_train, X_test, input_shape_seq),
                        ('RNN', build_rnn_model, X_train, X_test, input_shape_seq),
                        ('ANN', build_ann_model, X_train_flat, X_test_flat, input_shape_ann)
                    ]
                    
                    for model_name, model_builder, X_tr, X_te, input_shape in neural_models:
                        print(f"   🤖 Training {model_name} on Kalman-filtered Fed data...")
                        
                        try:
                            model = model_builder(input_shape)
                            history = model.fit(X_tr, y_train, epochs=100, batch_size=32, 
                                               validation_split=0.2, verbose=0, callbacks=[early_stopping])
                            
                            y_pred = model.predict(X_te, verbose=0).flatten()
                            metrics = calculate_comprehensive_metrics(y_test, y_pred)
                            models_results.append((model_name, metrics))
                            
                            print(f"      ✅ {model_name}: MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")
                            
                        except Exception as e:
                            print(f"      ❌ {model_name} failed: {e}")
                    
                    # Train ML baseline models
                    ml_results = train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test)
                    for ml_model, ml_metrics in ml_results.items():
                        models_results.append((ml_model.upper(), ml_metrics))
                    
                    # Store all results
                    for model_name, metrics in models_results:
                        result = {
                            'leading_indicators': combo,
                            'n_indicators': len(combo),
                            'scaler': scaler_name,
                            'model': model_name,
                            'seq_length': seq_length if model_name not in ['RANDOM_FOREST', 'XGBOOST'] else 'N/A',
                            'filter_applied': 'Kalman Multivariate',
                            'em_iterations': n_iter_em,
                            'burn_in': burn_in,
                            **metrics
                        }
                        results.append(result)
                
                except Exception as e:
                    print(f"   ❌ Experiment failed: {e}")
                    continue
    
    return results

# ================================================
# 13. EXECUTE ANALYSIS AND CREATE VISUALIZATIONS
# ================================================

if __name__ == "__main__":
    print("🚀 STARTING MULTIDIMENSIONAL KALMAN FILTER FED INFLATION FORECASTING ANALYSIS")
    
    try:
        # Run the analysis
        results = run_kalman_fed_inflation_forecasting()
        
        if not results:
            print("❌ No results generated!")
        else:
            # Convert to DataFrame
            results_df = pd.DataFrame(results)
            
            # Save results
            output_path = r'C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2 peer review\results\kalman_multiv_fed_inflation_forecasting_results.csv'
            results_df.to_csv(output_path, index=False)
            print(f"\n💾 Results saved to: {output_path}")
            
            # ================================================
            # 14. ENHANCED KALMAN FILTER VISUALIZATION
            # ================================================
            
            print(f"\n📊 CREATING MULTIDIMENSIONAL KALMAN FILTER ANALYSIS VISUALIZATIONS")
            print("-" * 50)
            
            # Create comprehensive visualization
            fig, axes = plt.subplots(2, 3, figsize=(20, 12))
            fig.suptitle('Multidimensional Kalman Filter Enhanced Inflation Forecasting\n(Fed Leading Indicators - Optimal State Estimation)', 
                         fontsize=16, fontweight='bold')
            
            # 1. Model Performance with Kalman Filtering (MSE)
            ax1 = axes[0, 0]
            model_mse = results_df.groupby('model')['mse'].mean().sort_values()
            bars1 = ax1.bar(range(len(model_mse)), model_mse.values, 
                            color=plt.cm.Set3(np.linspace(0, 1, len(model_mse))))
            ax1.set_xticks(range(len(model_mse)))
            ax1.set_xticklabels(model_mse.index, rotation=45, ha='right')
            ax1.set_ylabel('Mean Squared Error')
            ax1.set_title('Multivariate Kalman-Enhanced Fed Forecasting Performance')
            ax1.grid(True, alpha=0.3)
            
            # Add value labels
            for bar, value in zip(bars1, model_mse.values):
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f'{value:.4f}', ha='center', va='bottom', fontsize=9)
            
            # 2. R² with Kalman Filtering
            ax2 = axes[0, 1]
            model_r2 = results_df.groupby('model')['r2'].mean().sort_values(ascending=False)
            bars2 = ax2.bar(range(len(model_r2)), model_r2.values,
                            color=plt.cm.Set2(np.linspace(0, 1, len(model_r2))))
            ax2.set_xticks(range(len(model_r2)))
            ax2.set_xticklabels(model_r2.index, rotation=45, ha='right')
            ax2.set_ylabel('R² Score')
            ax2.set_title('Explained Variance (Multivariate Kalman-Filtered Fed Data)')
            ax2.grid(True, alpha=0.3)
            
            # 3. Fed Indicators Impact with Kalman
            ax3 = axes[0, 2]
            indicator_impact = results_df.groupby('n_indicators')['r2'].agg(['mean', 'std']).reset_index()
            ax3.errorbar(indicator_impact['n_indicators'], indicator_impact['mean'], 
                        yerr=indicator_impact['std'], marker='o', capsize=5, linewidth=2)
            ax3.set_xlabel('Number of Kalman-Filtered Fed Indicators')
            ax3.set_ylabel('R² Score (Mean ± Std)')
            ax3.set_title('Fed Indicator Complexity vs Performance')
            ax3.grid(True, alpha=0.3)
            
            # 4. Scaler Impact on Kalman-Filtered Fed Data
            ax4 = axes[1, 0]
            scaler_perf = results_df.groupby(['model', 'scaler'])['r2'].mean().unstack()
            scaler_perf.plot(kind='bar', ax=ax4, width=0.8)
            ax4.set_xlabel('Model Type')
            ax4.set_ylabel('R² Score')
            ax4.set_title('Normalization Impact (Post-Kalman Fed Data)')
            ax4.legend(title='Scaler')
            ax4.grid(True, alpha=0.3)
            plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            # 5. Top Kalman Fed Configurations
            ax5 = axes[1, 1]
            top_10 = results_df.nsmallest(10, 'mse')
            config_names = [f"{row['model']}\n{'+'.join(row['leading_indicators'][:2])}" 
                           for _, row in top_10.iterrows()]
            bars5 = ax5.barh(range(len(top_10)), top_10['mse'])
            ax5.set_yticks(range(len(top_10)))
            ax5.set_yticklabels(config_names, fontsize=8)
            ax5.set_xlabel('MSE')
            ax5.set_title('Top 10 Multivariate Kalman Fed Configurations')
            ax5.grid(True, alpha=0.3)
            
            # 6. MAPE Distribution for Kalman-Filtered Fed Models
            ax6 = axes[1, 2]
            results_df.boxplot(column='mape', by='model', ax=ax6)
            ax6.set_xlabel('Model Type')
            ax6.set_ylabel('MAPE (%)')
            ax6.set_title('Multivariate Kalman Fed Error Distribution')
            ax6.grid(True, alpha=0.3)
            plt.setp(ax6.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            plt.tight_layout()
            plt.savefig('multivariate_kalman_fed_inflation_forecasting_analysis.png', dpi=300, bbox_inches='tight')
            plt.show()
            
            # ================================================
            # 15. KALMAN FED SCIENTIFIC REPORT
            # ================================================
            
            print(f"\n📋 MULTIVARIATE KALMAN FILTER FED SCIENTIFIC ANALYSIS REPORT")
            print("="*70)
            
            print(f"\n🎯 RESEARCH OBJECTIVE:")
            print(f"Evaluate multivariate Kalman filtering for optimal state estimation in inflation forecasting")
            print(f"using Fed leading macroeconomic indicators (no data leakage)")
            
            print(f"\n🔧 MULTIVARIATE KALMAN FILTER METHODOLOGY:")
            print(f"   • Multivariate state space model with identity matrices")
            print(f"   • EM algorithm for optimal parameter estimation")
            print(f"   • Smoothing instead of filtering for retrospective analysis")
            print(f"   • Burn-in period handling for stability")
            
            print(f"\n📊 EXPERIMENTAL SETUP:")
            print(f"   Total experiments: {len(results_df)}")
            print(f"   Models tested: {results_df['model'].nunique()}")
            print(f"   Kalman-filtered Fed combinations: {results_df['leading_indicators'].nunique()}")
            print(f"   Best MSE achieved: {results_df['mse'].min():.6f}")
            print(f"   Best R² achieved: {results_df['r2'].max():.6f}")
            
            print(f"\n🏆 TOP 5 MULTIVARIATE KALMAN FED CONFIGURATIONS:")
            top_5 = results_df.nsmallest(5, 'mse')
            for i, (_, row) in enumerate(top_5.iterrows(), 1):
                indicators_str = '+'.join(row['leading_indicators'])
                print(f"{i}. {row['model']} | Kalman Fed States: {indicators_str}")
                print(f"   Scaler: {row['scaler']} | MSE: {row['mse']:.6f} | R²: {row['r2']:.4f}")
                print(f"   MAE: {row['mae']:.4f} | MAPE: {row['mape']:.2f}%")
                print("-" * 70)
            
            print(f"\n🤖 MULTIVARIATE KALMAN-ENHANCED FED MODEL RANKING:")
            model_ranking = results_df.groupby('model').agg({
                'mse': ['mean', 'std'],
                'r2': ['mean', 'std'],
                'mape': ['mean', 'std']
            }).round(6)
            print(model_ranking)
            
            print(f"\n📈 MULTIVARIATE KALMAN FED KEY FINDINGS:")
            best_config = results_df.loc[results_df['mse'].idxmin()]
            print(f"   • Best Kalman-enhanced model: {best_config['model']}")
            print(f"   • Optimal Fed state variables: {'+'.join(best_config['leading_indicators'])}")
            print(f"   • Best normalization: {best_config['scaler']}")
            
            print(f"\n💡 MULTIVARIATE KALMAN FILTER ADVANTAGES:")
            print(f"   • Cross-correlation handling between economic variables")
            print(f"   • Automatic parameter estimation via EM algorithm")
            print(f"   • Optimal smoothing for retrospective state estimation")
            print(f"   • Improved stability through burn-in handling")
            
            # ================================================
            # 16. COMPARISON WITH OTHER FILTERS
            # ================================================
            
            print(f"\n🔄 FILTER COMPARISON SUGGESTION:")
            print(f"Load and compare results with HP and BK filter outputs:")
            print(f"   1. hp_filter_results.csv")
            print(f"   2. bk_filter_results.csv")
            print(f"   3. kalman_multiv_fed_inflation_forecasting_results.csv (current)")
            
            print(f"\nCreate a consolidated comparison table for publication showing:")
            print(f"   • Average RMSE, MSE, R² and MAE for each filter type")
            print(f"   • Best performing model for each filter approach")
            print(f"   • Optimal indicator combinations across methods")
    
    except Exception as e:
        print(f"❌ Error during Multivariate Kalman Fed analysis: {e}")
        import traceback
        traceback.print_exc()

print(f"\n🎯 MULTIVARIATE KALMAN FILTER FED INFLATION FORECASTING COMPLETED!")
print("🔬 Optimal state estimation using multidimensional approach")
print("📊 Results enable direct comparison with HP and BK filter approaches")
print("🏦 Methodology applicable to central bank forecasting operations")